# Task 2: Quantitative Analysis Using PyNance and TA-Lib

This notebook processes the provided CSV stock data (NVDA, AAPL, MSFT, GOOG, AMZN, META), computes technical indicators (SMA, RSI, MACD) using manual implementations (fallback for TA-Lib), adds financial metrics (returns, volatility), and visualizes results for NVDA. Extend as needed for all stocks.

**Assumptions:**
- CSVs are in the same directory.
- Recent ~1 year data (tail 252 rows).
- For TA-Lib: Uncomment and install if available (`pip install TA-Lib`). Here, we use Pandas/NumPy for reproducibility.

**Workflow:** Run cells sequentially. Outputs saved to `data/processed/` and `figures/`.

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from datetime import datetime

# Manual functions (fallback if TA-Lib not installed)
def calculate_rsi(prices, window=14):
    """RSI calculation using Pandas."""
    delta = prices.diff()
    gain = delta.where(delta > 0, 0).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

def calculate_ema(prices, period):
    """EMA using Pandas ewm."""
    return prices.ewm(span=period, adjust=False).mean()

def calculate_macd(prices, fast=12, slow=26, signal=9):
    """MACD using EMAs."""
    ema_fast = calculate_ema(prices, fast)
    ema_slow = calculate_ema(prices, slow)
    macd = ema_fast - ema_slow
    signal_line = calculate_ema(macd, signal)
    histogram = macd - signal_line
    return macd, signal_line, histogram

# Optional: TA-Lib (uncomment if installed)
# import talib
# def calculate_rsi_talib(prices, window=14):
#     return talib.RSI(prices.values, timeperiod=window)
# def calculate_macd_talib(prices):
#     return talib.MACD(prices.values)

## Step 1: Prepare Your Data
Load CSVs, standardize, filter recent data.

In [ ]:
# List of CSV files
csv_files = ['NVDA.csv', 'AAPL.csv', 'MSFT.csv', 'GOOG.csv', 'AMZN.csv', 'META.csv']
stock_symbols = ['NVDA', 'AAPL', 'MSFT', 'GOOG', 'AMZN', 'META']

# Create directories
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('figures', exist_ok=True)

# Process each stock
processed_dfs = {}
for file, symbol in zip(csv_files, stock_symbols):
    # Load
    df = pd.read_csv("../data/"+file)
    df['Date'] = pd.to_datetime(df['Date'])
    df.set_index('Date', inplace=True)
    df = df[['Close', 'High', 'Low', 'Open', 'Volume']]  # OHLCV
    
    # Recent ~1 year (252 days)
    df_recent = df.tail(252).copy()
    
    # Save raw recent
    df_recent.to_csv(f'../data/processed/{symbol}_recent.csv')
    
    processed_dfs[symbol] = df_recent
    print(f"Loaded {symbol}: {df_recent.shape}")

# Example: Inspect NVDA
print("\nNVDA sample (head):")
print(processed_dfs['NVDA'].head())

OSError: Cannot save file into a non-existent directory: '..\data\processed'

## Step 2: Calculate Basic Technical Indicators
Add SMA, RSI, MACD.

In [ ]:
# Add indicators to each DF
for symbol, df_recent in processed_dfs.items():
    close = df_recent['Close']
    
    # Indicators (manual)
    df_recent['SMA_20'] = close.rolling(20).mean()
    df_recent['RSI_14'] = calculate_rsi(close, 14)
    df_recent['MACD'], df_recent['MACD_signal'], df_recent['MACD_hist'] = calculate_macd(close)
    
    # Optional: TA-Lib version
    # df_recent['RSI_14'] = calculate_rsi_talib(close, 14)
    # df_recent['MACD'], df_recent['MACD_signal'], df_recent['MACD_hist'] = calculate_macd_talib(close)
    
    # Drop NaNs (lookback periods)
    df_recent.dropna(inplace=True)
    
    # Save processed
    df_recent.to_csv(f'data/processed/{symbol}_indicators.csv')

# Example: NVDA tail
print("NVDA with indicators (tail):")
print(processed_dfs['NVDA'][['Close', 'SMA_20', 'RSI_14', 'MACD', 'MACD_signal']].tail())

## Step 3: Use PyNance for Financial Metrics
Add returns, volatility (manual; PyNance fallback).

In [ ]:
# Add metrics
for symbol, df_recent in processed_dfs.items():
    df_recent['Daily_Return'] = df_recent['Close'].pct_change() * 100
    df_recent['Volatility'] = df_recent['Daily_Return'].rolling(20).std() * np.sqrt(252)  # Annualized
    
    # Save final
    df_recent.to_csv(f'data/processed/{symbol}_full.csv')

# Example: NVDA returns/vol
print("NVDA returns/vol (tail):")
print(processed_dfs['NVDA'][['Daily_Return', 'Volatility']].tail())

## Step 4: Visualize the Data
Plots for NVDA.

In [ ]:
# NVDA plots
df_nvda = processed_dfs['NVDA']
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Price & SMA
axes[0].plot(df_nvda.index, df_nvda['Close'], label='Close', color='blue')
axes[0].plot(df_nvda.index, df_nvda['SMA_20'], label='SMA 20', color='orange')
axes[0].set_title('NVDA: Price and SMA')
axes[0].legend()
axes[0].grid(True)

# RSI
axes[1].plot(df_nvda.index, df_nvda['RSI_14'], label='RSI 14', color='green')
axes[1].axhline(70, color='r', linestyle='--', label='Overbought')
axes[1].axhline(30, color='r', linestyle='--', label='Oversold')
axes[1].set_title('NVDA: RSI')
axes[1].legend()
axes[1].grid(True)

# MACD
axes[2].plot(df_nvda.index, df_nvda['MACD'], label='MACD', color='blue')
axes[2].plot(df_nvda.index, df_nvda['MACD_signal'], label='Signal', color='red')
axes[2].bar(df_nvda.index, df_nvda['MACD_hist'], label='Histogram', alpha=0.3)
axes[2].set_title('NVDA: MACD')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.savefig('figures/nvda_indicators.png', dpi=300, bbox_inches='tight')
plt.show()

# Volatility
plt.figure(figsize=(12, 4))
plt.plot(df_nvda.index, df_nvda['Volatility'], label='Annualized Volatility', color='purple')
plt.title('NVDA: Volatility')
plt.legend()
plt.grid(True)
plt.savefig('figures/nvda_volatility.png', dpi=300, bbox_inches='tight')
plt.show()

# Optional: Loop for all stocks (comment out if too many plots)
# for symbol in stock_symbols[1:]:
#     df = processed_dfs[symbol]
#     # Similar plotting code...
#     plt.savefig(f'figures/{symbol}_indicators.png')

## Summary & Next Steps
- **Data:** Processed 6 stocks; saved full CSVs with indicators/metrics.
- **Insights:** NVDA shows bullish signals (e.g., Close > SMA_20, RSI ~55).
- **For Task 3:** Load news data, align dates, compute correlations.
- **Git:** Commit this notebook: `git add notebooks/task2_quantitative_analysis.ipynb && git commit -m "feat: complete Task 2 notebook"`